# 03 – Reproducible Data Preparation (Milestone 3, N05)
Chạy toàn bộ pipeline `src/prepare_data.py`: raw → interim → processed, kèm data dictionary, lineage, transformation log và validation summary.

In [1]:
import sys, json; sys.path.insert(0,'../src')
import pandas as pd
from config import *
import prepare_data
prepare_data.main()

[1] Load raw + kiểm tra SHA-256 và schema  7 bảng, studentVle 10.655.280 dòng
[2] Chọn phạm vi BBB+FFF, 4 kỳ: studentInfo 32,593 -> 15,671 
[2] Chọn phạm vi BBB+FFF, 4 kỳ: studentRegistration 32,593 -> 15,671 
[2] Chọn phạm vi BBB+FFF, 4 kỳ: assessments 206 -> 94 
[2] Chọn phạm vi BBB+FFF, 4 kỳ: vle 6,364 -> 3,121 
[2] Chọn phạm vi BBB+FFF, 4 kỳ: courses 22 -> 8 


[2] Chọn phạm vi BBB+FFF, 4 kỳ: studentVle 10,655,280 -> 5,582,063 
[2] Chọn phạm vi: studentAssessment (theo id_assessment của BBB+FFF) 173,912 -> 97,847 
[7] Ghép studentInfo + studentRegistration (1:1, validate) 15,671 -> 15,671 
[3] Chuẩn hóa nhãn imd_band '10-20' → '10-20%' (DQ-05)  
[5] imd_band thiếu → nhãn 'Unknown' + cờ; date_registration thiếu → giữ NaN + cờ (impute ở bước fit-on-train)  imd_band thiếu 430, date_registration thiếu 20
[6] Giữ final_result làm outcome chính thức; gắn cờ ca mâu thuẫn với date_unregistration (DQ-04)  Withdrawn không có ngày hủy 40, có ngày hủy nhưng không Withdrawn 6


[4] Giữ dòng trùng studentVle (nhiều lượt/ngày là hợp lệ), cộng dồn theo ngày  dòng trùng hoàn toàn 448.874; khóa nhiều dòng có click khác nhau 618.781


[9] Interim: tổng click theo sinh viên × ngày (toàn kỳ) 5,582,063 -> 865,577 


[8] Đặc trưng bài đánh giá đầu tiên (TMA1, hạn ngày 12–24): nộp trước ngày 28, đúng hạn, điểm (có điều kiện)  loại 268 bài bảo lưu
[7] Ghép đặc trưng VLE + đánh giá vào bảng đăng ký (left, 1:1) 15,671 -> 15,671 
[5] Không có bản ghi VLE trong 0–28 → 0 click + cờ no_vle_activity_0_28; days_since_last_active = 29 nếu chưa từng hoạt động  
[8] Đặc trưng dẫn xuất: clicks_change_w1_w4 = log1p(w4) − log1p(w1); log_clicks_0_28; highest_education_ord; registration_lead_days = −date_registration  
[5] Impute registration_lead_days bằng median tập train = 55 (fit trên train, áp cho validation/test)  
[6] Cờ chất lượng: late_registration_1_28 (cửa sổ quan sát bị rút ngắn), days_observable_0_28, flag_high_activity (> p99 tập train = 1910.96 click) – chỉ gắn cờ, không xóa/cắt  đăng ký muộn 117, hoạt động cao 115
[9] Population AQ1–AQ5: còn đăng ký ở cuối ngày 28 (C04, C06) 15,671 -> 13,020 


[9] Xuất 4 view: aq1_student_week, aq2_diagnostic_table, aq3_model_table (+ audit), context_all_registrations  


[8] Phân tích trùng lặp feature (train): 21 cặp |r| > 0,8; active_days_0_28 ≈ tổng 4 tuần (68.3% dòng bằng đúng); bộ rút gọn 21 feature, VIF lớn nhất 4.66  
done in 13.8s


# Transformation log

In [2]:
pd.DataFrame(json.load(open(META/'transformation_log.json')))

,step,action,rows_in,rows_out,note
0,1,Load raw + kiểm tra SHA-256 và schema,NaN,NaN,"7 bảng, studentVle 10.655.280 dòng"
1,2,"Chọn phạm vi BBB+FFF, 4 kỳ: studentInfo",32593.0,15671.0,
2,2,"Chọn phạm vi BBB+FFF, 4 kỳ: studentRegistration",32593.0,15671.0,
3,2,"Chọn phạm vi BBB+FFF, 4 kỳ: assessments",206.0,94.0,
4,2,"Chọn phạm vi BBB+FFF, 4 kỳ: vle",6364.0,3121.0,
5,2,"Chọn phạm vi BBB+FFF, 4 kỳ: courses",22.0,8.0,
6,2,"Chọn phạm vi BBB+FFF, 4 kỳ: studentVle",10655280.0,5582063.0,
7,2,Chọn phạm vi: studentAssessment (theo id_asses...,173912.0,97847.0,
8,7,"Ghép studentInfo + studentRegistration (1:1, v...",15671.0,15671.0,
9,3,Chuẩn hóa nhãn imd_band '10-20' → '10-20%' (DQ...,NaN,NaN,


# Validation summary (before/after, key, leakage, manual check)

In [3]:
V=json.load(open(META/'validation_summary.json'))
{k:V[k] for k in ['rows','key_unique_model_table','student_week_rows_equal_4x','leakage_columns_absent','feature_max_vle_date_used','fit_params','sensitivity_dedup']}

{'rows': {'registrations_scope': 15671,
  'population_active_d28': 13020,
  'student_week': 52080},
 'key_unique_model_table': True,
 'student_week_rows_equal_4x': True,
 'leakage_columns_absent': {'final_result': True,
  'label3': True,
  'date_unregistration': True,
  'unregistered_by_d28': True,
  'flag_withdrawn_no_unreg_date': True,
  'flag_unreg_date_not_withdrawn': True,
  'gender': True,
  'age_band': True,
  'disability': True,
  'imd_band': True,
  'region': True,
  'date_registration_missing': True},
 'feature_max_vle_date_used': 28,
 'fit_params': {'registration_lead_days_median_train': 55.0,
  'clicks_0_28_p99_train': 1910.9599999999919},
 'sensitivity_dedup': {'mean_clicks_keep': 332.5,
  'mean_clicks_dedup': 257.7,
  'spearman': 0.9959}}

In [4]:
pd.DataFrame(V['manual_check'])

,key,clicks_w1_w4_pipeline,clicks_w1_w4_manual,active_days_pipeline,active_days_manual,clicks_0_28_pipeline,clicks_0_28_manual,match
0,FFF-2013J-443753,"[0, 0, 0, 0]","[0, 0, 0, 0]",0,0,0,0,True
1,FFF-2014J-645518,"[200, 289, 320, 151]","[200, 289, 320, 151]",17,17,1037,1037,True
2,BBB-2014J-473095,"[0, 3, 27, 0]","[0, 3, 27, 0]",4,4,30,30,True
3,FFF-2013B-149680,"[115, 134, 217, 427]","[115, 134, 217, 427]",29,29,899,899,True
4,FFF-2013J-118050,"[193, 23, 54, 0]","[193, 23, 54, 0]",10,10,271,271,True
5,BBB-2013J-2617839,"[0, 15, 51, 16]","[0, 15, 51, 16]",6,6,82,82,True
6,FFF-2014B-2336654,"[30, 17, 0, 0]","[30, 17, 0, 0]",2,2,47,47,True
7,FFF-2014B-593220,"[0, 15, 0, 9]","[0, 15, 0, 9]",3,3,24,24,True
8,BBB-2014J-551276,"[0, 0, 0, 0]","[0, 0, 0, 0]",0,0,0,0,True
9,BBB-2014J-695972,"[0, 24, 1, 0]","[0, 24, 1, 0]",2,2,25,25,True


In [5]:
pd.DataFrame(V['target_rate']['by_split_active_d28']).T

,size,mean
test,3644.0,0.3776
train,6795.0,0.4481
validation,2581.0,0.4649


# Prepared views

In [6]:
for f in sorted(PROCESSED.glob('*.csv')):
    d=pd.read_csv(f); print(f.name, d.shape)
pd.read_csv(PROCESSED/'aq3_model_table.csv').head()

aq1_student_week.csv (52080, 9)
aq2_diagnostic_table.csv (13020, 45)
aq3_audit_attributes.csv (13020, 13)
aq3_model_table.csv (13020, 41)
context_all_registrations.csv (15671, 12)


,code_module,code_presentation,id_student,split,clicks_w1,clicks_w2,clicks_w3,clicks_w4,active_days_w1,active_days_w2,...,num_of_prev_attempts,studied_credits,highest_education_ord,registration_lead_days,tma1_score,seen_in_earlier_presentation,late_registration_1_28,days_observable_0_28,flag_high_activity,target_noncompletion
0,BBB,2013B,23629,train,23,5,9,2,2,1,...,2,60,1,47.0,67.0,0,0,29,0,1
1,BBB,2013B,25107,train,10,0,66,3,1,0,...,0,120,1,73.0,59.0,0,0,29,0,0
2,BBB,2013B,27891,train,124,18,16,13,3,1,...,2,120,1,58.0,73.0,0,0,29,0,1
3,BBB,2013B,29144,train,126,0,74,0,5,0,...,0,120,1,24.0,60.0,0,0,29,0,1
4,BBB,2013B,31663,train,302,97,188,238,7,6,...,0,60,2,60.0,88.0,0,0,29,0,0
